# Lag Statistics Summary

This notebook summarizes lag usage for an SNN run using the saved CSV exports only.

It covers:

- lag distribution per block
- active-weight histogram per lag
- per-lag mean absolute active weight
- per-lag selection rate and weight magnitude summary

It reads the saved `lag_selections/*.csv` artifacts and does not rebuild or run the model.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / 'model_training').exists():
            return path
    raise FileNotFoundError('Could not find repo root containing model_training/')


REPO_ROOT = find_repo_root()
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)


In [ ]:
# Point this at the run you want to inspect.
RUN_DIR = REPO_ROOT / 'model_training' / 'logs' / 'snn' / '2026-04-05_00-26-06_065913_snn_pid261059'

print(f'Run dir: {RUN_DIR}')


In [ ]:
def resolve_lag_dir(run_dir: Path) -> Path:
    candidates = [
        run_dir / 'lag_selections',
        run_dir / 'notebook_analysis' / 'trained' / 'output' / 'lag_selections',
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find lag selection exports under {run_dir}')


def load_distribution_df(run_dir: Path) -> pd.DataFrame:
    lag_dir = resolve_lag_dir(run_dir)
    frames = []
    for path in sorted(lag_dir.glob('block_*_lag_distribution.csv')):
        block_idx = int(path.stem.split('_')[1])
        df = pd.read_csv(path)
        df['block_idx'] = block_idx
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f'No lag-distribution CSVs found under {lag_dir}')
    return pd.concat(frames, ignore_index=True)


def load_weight_df(run_dir: Path) -> pd.DataFrame:
    lag_dir = resolve_lag_dir(run_dir)
    frames = []
    for path in sorted(lag_dir.glob('block_*_lag_weights.csv')):
        block_idx = int(path.stem.split('_')[1])
        df = pd.read_csv(path)
        df['block_idx'] = block_idx
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f'No lag-weight CSVs found under {lag_dir}')
    weight_df = pd.concat(frames, ignore_index=True)
    weight_df['selected'] = weight_df['selected'].astype(bool)
    weight_df['abs_weight'] = weight_df['weight'].abs()
    return weight_df


def mean_abs_active_weight_by_lag(active_weight_df: pd.DataFrame) -> pd.DataFrame:
    return (
        active_weight_df.groupby(['block_idx', 'lag'], as_index=False)['abs_weight']
        .mean()
        .rename(columns={'abs_weight': 'mean_abs_active_weight'})
        .sort_values(['block_idx', 'lag'])
        .reset_index(drop=True)
    )


def build_selection_summary(weight_df: pd.DataFrame) -> pd.DataFrame:
    return (
        weight_df.groupby(['block_idx', 'lag'], as_index=False)
        .agg(
            num_pairs=('selected', 'size'),
            num_selected_pairs=('selected', 'sum'),
            mean_abs_weight_all=('abs_weight', 'mean'),
            mean_abs_weight_selected=('abs_weight', lambda s: float(s[s > 0].mean()) if np.any(s > 0) else 0.0),
        )
        .assign(selection_rate=lambda df: df['num_selected_pairs'] / df['num_pairs'])
        .sort_values(['block_idx', 'lag'])
        .reset_index(drop=True)
    )


In [ ]:
lag_dir = resolve_lag_dir(RUN_DIR)
distribution_df = load_distribution_df(RUN_DIR)
weight_df = load_weight_df(RUN_DIR)
active_weight_df = weight_df[weight_df['selected']].copy()
block_ids = sorted(distribution_df['block_idx'].unique().tolist())
n_blocks = len(block_ids)
mean_abs_weight_df = mean_abs_active_weight_by_lag(active_weight_df)
selection_summary_df = build_selection_summary(weight_df)

print(f'Lag export dir: {lag_dir}')
print(f'Blocks found: {block_ids}')
display(distribution_df.head())
display(active_weight_df.head())
display(selection_summary_df.head())


In [ ]:
fig, axes = plt.subplots(1, n_blocks, figsize=(5 * n_blocks, 4), squeeze=False, sharey=True)
axes = axes[0]

for ax, block_idx in zip(axes, block_ids):
    block_df = distribution_df[distribution_df['block_idx'] == block_idx].sort_values('lag')
    ax.bar(block_df['lag'].astype(str), block_df['count'], color='#4C78A8', alpha=0.9)
    ax.set_title(f'Block {block_idx}')
    ax.set_xlabel('Lag')
    ax.set_ylabel('Count')

fig.suptitle('Lag Distribution Per Block', fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
for block_idx in block_ids:
    block_df = active_weight_df[active_weight_df['block_idx'] == block_idx].copy()
    lag_order = sorted(block_df['lag'].unique())
    n_lags = len(lag_order)

    fig, axes = plt.subplots(1, n_lags, figsize=(4.5 * n_lags, 4.5), squeeze=False, sharey=True)
    axes = axes[0]

    for ax, lag in zip(axes, lag_order):
        values = block_df.loc[block_df['lag'] == lag, 'weight'].to_numpy()
        bins = min(40, max(10, int(np.sqrt(len(values)))))
        ax.hist(values, bins=bins, color='#F58518', alpha=0.85, edgecolor='white')
        ax.set_title(f'Lag {lag}')
        ax.set_xlabel('Active Weight')
        ax.set_ylabel('Count')
        ax.axvline(0.0, color='black', linewidth=1.0, linestyle='--', alpha=0.7)

    fig.suptitle(f'Block {block_idx}: Active Weight Histograms Per Lag', fontsize=14)
    fig.tight_layout()
    plt.show()


In [ ]:
display(mean_abs_weight_df)
display(selection_summary_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for block_idx in block_ids:
    block_df = selection_summary_df[selection_summary_df['block_idx'] == block_idx].sort_values('lag')
    axes[0].plot(block_df['lag'], block_df['selection_rate'], marker='o', label=f'Block {block_idx}')
    axes[1].plot(block_df['lag'], block_df['mean_abs_weight_selected'], marker='o', label=f'Block {block_idx}')

axes[0].set_title('Selection Rate By Lag')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('Selected pairs / total pairs')
axes[0].set_ylim(0.0, 1.05)
axes[1].set_title('Mean |Active Weight| By Lag')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('Mean |weight| over selected pairs')
axes[1].legend(loc='best')

fig.tight_layout()
plt.show()
